# Wine Reviews Data Cleaning

In [1]:
import numpy as np
import pandas as pd

## Load

In [2]:
df = pd.read_parquet("data_clean/winemag-data-130k-v2.parquet")
df.shape

(119988, 13)

## Repeated reviews

In [3]:
df = df.drop_duplicates("description", keep=False).reset_index(drop=True)
df.shape

(119922, 13)

## Taster Twitter handle

In [4]:
df = df.drop(columns="taster_twitter_handle")

## Vintage year

In [5]:
after_winery = pd.Series([t[len(w):] for t, w in zip(df["title"], df["winery"])])
df["vintage"] = after_winery.str.extract(r"^\s*(19\d{2}|20[01]\d)\b")[0].astype("Int64")
df["vintage"].describe()

count       115633.0
mean     2010.587064
std           3.6846
min           1934.0
25%           2009.0
50%           2011.0
75%           2013.0
max           2017.0
Name: vintage, dtype: Float64

## Price

In [6]:
bad_price = df["title"].str.startswith(("Château les Ormes Sorbet 2013", "Blair 2013 Roger Rose"))
df.loc[bad_price, "price"] = np.nan
df["log_price"] = np.log(df["price"])
df["price_missing"] = df["price"].isna()
df["price_missing"].sum()

np.int64(8387)

## Countries

In [7]:
counts = df["country"].value_counts()
rare = counts[counts < 50].index
df["country_group"] = df["country"].where(~df["country"].isin(rare), "Other")
df["country_group"].value_counts()

country_group
US              50440
France          20349
Italy           17902
Spain            6116
Portugal         5256
Chile            4184
Argentina        3542
Austria          3032
Australia        2196
Germany          1992
South Africa     1301
New Zealand      1278
Israel            466
Greece            430
Canada            226
Other             201
Bulgaria          132
Hungary           129
Romania           102
Uruguay            98
Turkey             81
Slovenia           77
Georgia            76
Croatia            70
Mexico             68
England            63
Moldova            56
Name: count, dtype: int64

In [8]:
df.to_parquet("data_clean/wine_reviews_clean.parquet", index=False)
df.shape

(119922, 16)